In [ ]:
# Example 1: Basic TensorDict Creation and Operations
from tensordict import TensorDict
import torch

print("Basic TensorDict Creation and Operations:")

try:
    # Create a TensorDict with multiple tensors
    batch_size = 32
    
    data = TensorDict({
        "observations": torch.randn(batch_size, 84, 84, 3),  # Image observations
        "actions": torch.randint(0, 4, (batch_size,)),       # Discrete actions
        "rewards": torch.randn(batch_size),                  # Scalar rewards
        "done": torch.randint(0, 2, (batch_size,)).bool(),  # Terminal flags
        "info": {
            "episode_length": torch.randint(1, 100, (batch_size,)),
            "score": torch.randn(batch_size)
        }
    }, batch_size=[batch_size])
    
    print(f"✅ Created TensorDict with batch size: {data.batch_size}")
    print(f"Keys: {list(data.keys())}")
    print(f"Nested keys: {list(data['info'].keys())}")
    
    # Basic operations
    print(f"\nBasic operations:")
    print(f"Total number of elements: {data.numel()}")
    # API drift: `.nbytes` -> `.bytes()` (a method, not a property).
    print(f"Memory usage: {data.bytes() / 1024 / 1024:.2f} MB")
    print(f"Device: {data.device}")
    
    # Indexing operations
    first_sample = data[0]
    print(f"\nFirst sample shapes:")
    print(f"  observations: {first_sample['observations'].shape}")
    print(f"  actions: {first_sample['actions'].shape}")
    print(f"  rewards: {first_sample['rewards'].shape}")
    
    # Batch slicing
    subset = data[10:20]
    print(f"\nSubset batch size: {subset.batch_size}")
    
    # Adding new fields
    data["values"] = torch.randn(batch_size)
    print(f"Added 'values' field. New keys: {list(data.keys())}")
    
    # Clone and modify
    data_copy = data.clone()
    data_copy["rewards"] *= 2.0
    print(f"Original rewards mean: {data['rewards'].mean():.3f}")
    print(f"Modified rewards mean: {data_copy['rewards'].mean():.3f}")

except Exception as e:
    print(f"⚠️ TensorDict demonstration: {e}")
    print("Note: Would create dictionary-like tensor containers with batch consistency")
    raise AssertionError(
        f"tensordict example (cell 0) failed: {type(e).__name__}: {e}"
    ) from e

print("✅ Basic TensorDict creation and operations completed")

## Example 2: Batch Operations and Data Pipeline Processing

In [ ]:
# Example 2: Batch Operations and Data Pipeline Processing
from tensordict import TensorDict
import torch

print("Batch Operations and Data Pipeline Processing:")

try:
    # Create multiple TensorDict batches for pipeline processing
    def create_batch(batch_id, batch_size=16):
        """Create a sample batch for processing"""
        return TensorDict({
            "batch_id": torch.full((batch_size,), batch_id),
            "features": torch.randn(batch_size, 128),
            "labels": torch.randint(0, 10, (batch_size,)),
            "metadata": {
                "timestamp": torch.full((batch_size,), batch_id * 1000),
                "source": torch.randint(0, 3, (batch_size,))
            }
        }, batch_size=[batch_size])
    
    # Generate multiple batches
    batches = [create_batch(i) for i in range(5)]
    print(f"✅ Created {len(batches)} batches")
    
    # Concatenate batches
    combined_batch = torch.cat(batches, dim=0)
    print(f"Combined batch size: {combined_batch.batch_size}")
    print(f"Total samples: {combined_batch.batch_size[0]}")
    
    # Apply transformations to entire batch
    print(f"\nApplying batch transformations:")
    
    # Normalize features
    combined_batch["features_normalized"] = (
        combined_batch["features"] - combined_batch["features"].mean(dim=1, keepdim=True)
    ) / combined_batch["features"].std(dim=1, keepdim=True)
    
    # One-hot encode labels
    num_classes = 10
    combined_batch["labels_onehot"] = torch.nn.functional.one_hot(
        combined_batch["labels"], num_classes=num_classes
    ).float()
    
    # Add computed features
    combined_batch["feature_stats"] = {
        "mean": combined_batch["features"].mean(dim=1),
        "std": combined_batch["features"].std(dim=1),
        "max": combined_batch["features"].max(dim=1)[0]
    }
    
    print(f"Added normalized features: {combined_batch['features_normalized'].shape}")
    print(f"Added one-hot labels: {combined_batch['labels_onehot'].shape}")
    print(f"Added feature statistics: {list(combined_batch['feature_stats'].keys())}")
    
    # Batch filtering and selection
    print(f"\nBatch filtering operations:")
    
    # Filter by label
    class_0_mask = combined_batch["labels"] == 0
    class_0_samples = combined_batch[class_0_mask]
    print(f"Class 0 samples: {class_0_samples.batch_size[0]} out of {combined_batch.batch_size[0]}")
    
    # Filter by feature statistics
    high_variance_mask = combined_batch["feature_stats"]["std"] > 1.0
    high_variance_samples = combined_batch[high_variance_mask]
    print(f"High variance samples: {high_variance_samples.batch_size[0]}")
    
    # Split data into train/validation
    n_samples = combined_batch.batch_size[0]
    indices = torch.randperm(n_samples)
    train_size = int(0.8 * n_samples)
    
    train_indices = indices[:train_size]
    val_indices = indices[train_size:]
    
    train_data = combined_batch[train_indices]
    val_data = combined_batch[val_indices]
    
    print(f"\nData split:")
    print(f"  Training samples: {train_data.batch_size[0]}")
    print(f"  Validation samples: {val_data.batch_size[0]}")
    
    # Batch aggregation operations
    print(f"\nBatch aggregations:")
    
    # Compute batch statistics
    batch_stats = {
        "mean_features": combined_batch["features"].mean(dim=0),
        "std_features": combined_batch["features"].std(dim=0),
        "label_distribution": torch.bincount(combined_batch["labels"]),
        "feature_correlations": torch.corrcoef(combined_batch["features"].T)[:5, :5]  # Sample 5x5
    }
    
    print(f"  Feature means shape: {batch_stats['mean_features'].shape}")
    print(f"  Label distribution: {batch_stats['label_distribution'].tolist()}")
    print(f"  Feature correlation matrix: {batch_stats['feature_correlations'].shape}")

except Exception as e:
    print(f"⚠️ Batch operations demonstration: {e}")
    print("Note: Would perform efficient batch processing and data pipeline operations")
    raise AssertionError(
        f"tensordict example (cell 2) failed: {type(e).__name__}: {e}"
    ) from e

print("✅ Batch operations and data pipeline processing completed")

## Example 3: Device Management and Memory Optimization

In [ ]:
# Example 3: Device Management and Memory Optimization
from tensordict import TensorDict
import torch

print("Device Management and Memory Optimization:")

try:
    # Create TensorDict on CPU
    cpu_data = TensorDict({
        "images": torch.randn(64, 3, 224, 224),
        "labels": torch.randint(0, 1000, (64,)),
        "features": torch.randn(64, 512),
        "metadata": {
            "batch_idx": torch.arange(64),
            "augmented": torch.randint(0, 2, (64,)).bool()
        }
    }, batch_size=[64])
    
    print(f"✅ Created TensorDict on device: {cpu_data.device}")
    print(f"Memory usage (CPU): {cpu_data.bytes() / 1024 / 1024:.2f} MB")
    
    device_data = cpu_data
    
    # Memory optimization techniques
    print(f"\nMemory optimization:")
    
    # Use different data types for memory efficiency
    optimized_data = TensorDict({
        "images_fp16": device_data["images"].half(),  # Half precision for images
        "labels": device_data["labels"].long(),       # Long for indices
        "features_fp16": device_data["features"].half(),
        "flags": device_data["metadata"]["augmented"].bool()  # Boolean for flags
    }, batch_size=[64])
    
    # Memory comparison
    original_memory = device_data.bytes() / 1024 / 1024
    optimized_memory = optimized_data.bytes() / 1024 / 1024
    memory_savings = ((original_memory - optimized_memory) / original_memory) * 100
    
    print(f"Original memory: {original_memory:.2f} MB")
    print(f"Optimized memory: {optimized_memory:.2f} MB")
    print(f"Memory savings: {memory_savings:.1f}%")
    
    # In-place operations for memory efficiency
    print(f"\nIn-place operations:")
    
    # Clone for testing in-place operations
    inplace_data = device_data.clone()
    
    # In-place normalization
    inplace_data["features"].sub_(inplace_data["features"].mean(dim=1, keepdim=True))
    inplace_data["features"].div_(inplace_data["features"].std(dim=1, keepdim=True))
    
    # In-place image preprocessing
    inplace_data["images"].clamp_(0, 1)  # Clamp to [0, 1]
    inplace_data["images"].mul_(2).sub_(1)  # Scale to [-1, 1]
    
    print(f"Applied in-place transformations")
    print(f"Features mean: {inplace_data['features'].mean():.6f}")
    print(f"Images range: [{inplace_data['images'].min():.3f}, {inplace_data['images'].max():.3f}]")
    
    # Memory sharing and views
    print(f"\nMemory sharing with views:")
    
    # Create views that share memory
    image_view = device_data["images"].view(64, -1)  # Flatten images
    subset_view = device_data[::2]  # Every other sample
    
    print(f"Original images shape: {device_data['images'].shape}")
    print(f"Flattened view shape: {image_view.shape}")
    print(f"Subset batch size: {subset_view.batch_size}")
    print(f"Memory shared: {image_view.data_ptr() == device_data['images'].data_ptr()}")
    
    # Contiguous memory optimization
    print(f"\nContiguous memory optimization:")
    
    # Check if tensors are contiguous
    images_contiguous = device_data["images"].is_contiguous()
    print(f"Images contiguous: {images_contiguous}")
    
    if not images_contiguous:
        device_data["images"] = device_data["images"].contiguous()
        print("Made images contiguous")
    
except Exception as e:
    print(f"⚠️ Device management demonstration: {e}")
    print("Note: Would handle device transfers and memory optimization for efficient processing")
    raise AssertionError(
        f"tensordict example (cell 4) failed: {type(e).__name__}: {e}"
    ) from e

print("✅ Device management and memory optimization completed")

In [ ]:
# The device half of Example 3. tensordict's whole claim is that ONE `.to(device)` moves every leaf of
# an arbitrarily nested structure, so that is asserted here rather than printed.
import torch
from tensordict import TensorDict

assert torch.cuda.is_available(), (
    f"requires-gpu cell ran but torch.cuda.is_available() is False — torch {torch.__version__}, "
    f"built against CUDA {torch.version.cuda}, device_count={torch.cuda.device_count()}"
)

host_data = TensorDict({
    "images": torch.randn(8, 3, 32, 32),
    "features": torch.randn(8, 512),
    "metadata": {"batch_idx": torch.arange(8)},
}, batch_size=[8])

moved = host_data.to("cuda")

# The nested leaf is the one that matters: a shallow implementation moves the top level and leaves
# `metadata` behind on the host.
assert moved["images"].device.type == "cuda", moved["images"].device
assert moved["features"].device.type == "cuda", moved["features"].device
assert moved["metadata"]["batch_idx"].device.type == "cuda", moved["metadata"]["batch_idx"].device

# ...and the round trip is lossless.
assert torch.equal(moved["images"].cpu(), host_data["images"])

print(f"✅ TensorDict moved every nested leaf to {moved['images'].device}")

## Example 4: Neural Network Integration with TensorDictModule

In [ ]:
# Example 4: Neural Network Integration with TensorDictModule
from tensordict import TensorDict
from tensordict.nn import TensorDictModule, TensorDictSequential
import torch
import torch.nn as nn

print("Neural Network Integration with TensorDictModule:")

try:
    # Create sample input data
    batch_size = 32
    input_data = TensorDict({
        "observations": torch.randn(batch_size, 64),
        "actions": torch.randn(batch_size, 8),
        "context": torch.randn(batch_size, 16)
    }, batch_size=[batch_size])
    
    print(f"✅ Created input TensorDict with batch size: {input_data.batch_size}")
    
    # Define neural network components using TensorDictModule
    print(f"\nCreating TensorDictModule components:")
    
    # Feature encoder
    feature_encoder = TensorDictModule(
        nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
        ),
        in_keys=["observations"],
        out_keys=["encoded_obs"]
    )
    
    # Action processor
    action_processor = TensorDictModule(
        nn.Sequential(
            nn.Linear(8, 32),
            nn.Tanh(),
            nn.Linear(32, 16)
        ),
        in_keys=["actions"],
        out_keys=["processed_actions"]
    )
    
    # Context embedder
    context_embedder = TensorDictModule(
        nn.Linear(16, 32),
        in_keys=["context"],
        out_keys=["context_embedding"]
    )
    
    # Fusion network
    # A TensorDictModule with N `in_keys` calls its module with N POSITIONAL arguments, and
    # `nn.Sequential.forward` takes exactly one — hence
    # `TypeError: Sequential.forward() takes 2 positional arguments but 4 were given`.
    # Fusing several inputs therefore needs a module that concatenates them first; that
    # concatenation is the fusion, and leaving it implicit is what broke this example.
    class ConcatFusion(nn.Module):
        def __init__(self, tail):
            super().__init__()
            self.tail = tail

        def forward(self, *inputs):
            return self.tail(torch.cat(inputs, dim=-1))

    fusion_network = TensorDictModule(
        ConcatFusion(
            nn.Sequential(
                nn.Linear(64 + 16 + 32, 128),  # encoded_obs + processed_actions + context_embedding
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Linear(64, 32)
            )
        ),
        in_keys=["encoded_obs", "processed_actions", "context_embedding"],
        out_keys=["fused_features"]
    )
    
    print(f"  Feature encoder: observations -> encoded_obs")
    print(f"  Action processor: actions -> processed_actions")
    print(f"  Context embedder: context -> context_embedding")
    print(f"  Fusion network: [encoded_obs, processed_actions, context_embedding] -> fused_features")
    
    # Create sequential pipeline
    print(f"\nBuilding TensorDictSequential pipeline:")
    
    pipeline = TensorDictSequential(
        feature_encoder,
        action_processor,
        context_embedder,
        fusion_network
    )
    
    # Forward pass through pipeline
    print(f"\nExecuting forward pass:")
    
    output_data = pipeline(input_data)
    
    print(f"Input keys: {list(input_data.keys())}")
    print(f"Output keys: {list(output_data.keys())}")
    print(f"Fused features shape: {output_data['fused_features'].shape}")
    
    # Add prediction heads
    print(f"\nAdding prediction heads:")
    
    # Value prediction head
    value_head = TensorDictModule(
        nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        ),
        in_keys=["fused_features"],
        out_keys=["value_prediction"]
    )
    
    # Policy prediction head
    policy_head = TensorDictModule(
        nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8),  # 8 action dimensions
            nn.Softmax(dim=-1)
        ),
        in_keys=["fused_features"],
        out_keys=["policy_logits"]
    )
    
    # Create complete model
    complete_model = TensorDictSequential(
        pipeline,
        value_head,
        policy_head
    )
    
    # Full forward pass
    final_output = complete_model(input_data)
    
    print(f"Value predictions shape: {final_output['value_prediction'].shape}")
    print(f"Policy logits shape: {final_output['policy_logits'].shape}")
    print(f"Policy logits sum (should be ~1): {final_output['policy_logits'].sum(dim=-1).mean():.3f}")
    
    # Training simulation
    print(f"\nTraining simulation:")
    
    # Add target values for training
    targets = TensorDict({
        "target_value": torch.randn(batch_size, 1),
        "target_actions": torch.randint(0, 8, (batch_size,))
    }, batch_size=[batch_size])
    
    # Compute losses
    value_loss = nn.MSELoss()(final_output["value_prediction"], targets["target_value"])
    policy_loss = nn.CrossEntropyLoss()(final_output["policy_logits"], targets["target_actions"])
    
    total_loss = value_loss + policy_loss
    
    print(f"Value loss: {value_loss.item():.4f}")
    print(f"Policy loss: {policy_loss.item():.4f}")
    print(f"Total loss: {total_loss.item():.4f}")
    
    # Parameter counting
    total_params = sum(p.numel() for p in complete_model.parameters())
    trainable_params = sum(p.numel() for p in complete_model.parameters() if p.requires_grad)
    
    print(f"\nModel statistics:")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Model size: {total_params * 4 / 1024 / 1024:.2f} MB (float32)")

except Exception as e:
    print(f"⚠️ Neural network integration demonstration: {e}")
    print("Note: Would integrate TensorDict with neural networks for structured data flow")
    raise AssertionError(
        f"tensordict example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

print("✅ Neural network integration with TensorDictModule completed")

## Example 5: Advanced Data Structures and Reinforcement Learning Integration

In [ ]:
# Example 5: Advanced Data Structures and Reinforcement Learning Integration
from tensordict import TensorDict
import torch
from collections import deque

print("Advanced Data Structures and Reinforcement Learning Integration:")

try:
    # Create trajectory data for RL
    print("Creating RL trajectory data:")
    
    sequence_length = 100
    batch_size = 8
    
    # Create timestep data
    def create_timestep(t, batch_size):
        return TensorDict({
            "observation": torch.randn(batch_size, 84, 84, 4),  # Stacked frames
            "action": torch.randint(0, 6, (batch_size,)),       # Discrete actions
            "reward": torch.randn(batch_size),                  # Scalar rewards
            "done": torch.rand(batch_size) < 0.02,              # Terminal flags (~2% done)
            "value": torch.randn(batch_size),                   # Value estimates
            "log_prob": torch.randn(batch_size),                # Action log probabilities
            "advantage": torch.randn(batch_size),               # Advantage estimates
            "timestep": torch.full((batch_size,), t, dtype=torch.long)
        }, batch_size=[batch_size])
    
    # Generate sequence of timesteps
    trajectory = TensorDict({}, batch_size=[sequence_length, batch_size])
    
    for t in range(sequence_length):
        timestep_data = create_timestep(t, batch_size)
        for key, value in timestep_data.items():
            if key not in trajectory:
                # Initialize with zeros, then fill
                if key == "observation":
                    trajectory[key] = torch.zeros(sequence_length, batch_size, 84, 84, 4)
                elif key == "done":
                    trajectory[key] = torch.zeros(sequence_length, batch_size, dtype=torch.bool)
                elif key == "timestep":
                    trajectory[key] = torch.zeros(sequence_length, batch_size, dtype=torch.long)
                else:
                    trajectory[key] = torch.zeros(sequence_length, batch_size)
            
            trajectory[key][t] = value
    
    print(f"✅ Created trajectory: {trajectory.batch_size}")
    print(f"Keys: {list(trajectory.keys())}")
    
    # Experience replay buffer simulation
    print(f"\nExperience replay buffer:")
    
    class TensorDictReplayBuffer:
        def __init__(self, capacity, sample_batch_size):
            self.capacity = capacity
            self.sample_batch_size = sample_batch_size
            self.buffer = deque(maxlen=capacity)
            self.position = 0
        
        def push(self, transition):
            """Add a transition to the buffer"""
            if len(self.buffer) < self.capacity:
                self.buffer.append(transition)
            else:
                self.buffer[self.position] = transition
                self.position = (self.position + 1) % self.capacity
        
        def sample(self):
            """Sample a batch of transitions"""
            if len(self.buffer) < self.sample_batch_size:
                return None
            
            # Random sampling
            indices = torch.randint(0, len(self.buffer), (self.sample_batch_size,))
            batch_transitions = [self.buffer[i] for i in indices]
            
            # Stack transitions into batch
            batch = TensorDict.stack(batch_transitions, dim=0)
            return batch
        
        def __len__(self):
            return len(self.buffer)
    
    # Create and populate replay buffer
    replay_buffer = TensorDictReplayBuffer(capacity=1000, sample_batch_size=32)
    
    # Add transitions to buffer
    for t in range(50):
        transition = create_timestep(t, 1)  # Single transition
        replay_buffer.push(transition.squeeze(0))  # Remove batch dimension
    
    print(f"Replay buffer size: {len(replay_buffer)}")
    
    # Sample from buffer
    sampled_batch = replay_buffer.sample()
    if sampled_batch is not None:
        print(f"Sampled batch size: {sampled_batch.batch_size}")
    
    # Advanced indexing and manipulation
    print(f"\nAdvanced indexing operations:")
    
    # Create multi-dimensional tensor dict
    multi_env_data = TensorDict({
        "observations": torch.randn(4, 8, 84, 84, 3),  # 4 environments, 8 agents
        "actions": torch.randint(0, 5, (4, 8)),
        "rewards": torch.randn(4, 8),
        "agent_info": {
            "health": torch.rand(4, 8),
            "position": torch.randn(4, 8, 2),
            "inventory": torch.randint(0, 10, (4, 8, 5))
        }
    }, batch_size=[4, 8])
    
    print(f"Multi-environment data: {multi_env_data.batch_size}")
    
    # Environment-wise operations
    env_0_data = multi_env_data[0]  # All agents in environment 0
    print(f"Environment 0 data: {env_0_data.batch_size}")
    
    # Agent-wise operations across environments
    agent_0_data = multi_env_data[:, 0]  # Agent 0 across all environments
    print(f"Agent 0 across environments: {agent_0_data.batch_size}")
    
    # Fancy indexing
    high_reward_mask = multi_env_data["rewards"] > 0
    high_reward_agents = multi_env_data[high_reward_mask]
    print(f"High reward agents: {high_reward_agents.batch_size}")
    
    # Temporal operations
    print(f"\nTemporal operations:")
    
    # Calculate returns (discounted cumulative rewards)
    gamma = 0.99
    returns = torch.zeros_like(trajectory["reward"])
    
    # Backward pass to compute returns
    for t in reversed(range(sequence_length)):
        if t == sequence_length - 1:
            returns[t] = trajectory["reward"][t]
        else:
            returns[t] = trajectory["reward"][t] + gamma * returns[t + 1] * (~trajectory["done"][t])
    
    trajectory["returns"] = returns
    
    # Calculate advantages (returns - values)
    trajectory["advantage"] = trajectory["returns"] - trajectory["value"]
    
    print(f"Added returns and advantages")
    print(f"Mean return: {trajectory['returns'].mean():.3f}")
    print(f"Mean advantage: {trajectory['advantage'].mean():.3f}")
    
    # Episode segmentation
    print(f"\nEpisode segmentation:")
    
    # Find episode boundaries
    done_indices = torch.where(trajectory["done"].any(dim=1))[0]
    print(f"Episode endings at timesteps: {done_indices.tolist()[:5]}...")  # First 5
    
    # Create episode-wise data
    episode_lengths = []
    start_idx = 0
    
    for done_idx in done_indices[:3]:  # Process first 3 episodes
        episode_length = done_idx.item() - start_idx + 1
        episode_lengths.append(episode_length)
        
        episode_data = trajectory[start_idx:done_idx + 1]
        episode_return = episode_data["returns"][0].mean().item()  # Return from start
        
        print(f"Episode length: {episode_length}, Return: {episode_return:.3f}")
        start_idx = done_idx.item() + 1
    
    print(f"Average episode length: {torch.tensor(episode_lengths).float().mean():.1f}")

except Exception as e:
    print(f"⚠️ Advanced data structures demonstration: {e}")
    print("Note: Would implement complex RL data structures and temporal operations")
    raise AssertionError(
        f"tensordict example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e

print("✅ Advanced data structures and reinforcement learning integration completed")

## Summary

This notebook demonstrated 5 comprehensive examples of the `tensordict` package:

1. **Basic TensorDict Creation and Operations** - Dictionary-like tensor containers with consistent batch dimensions, indexing, and nested data structures
2. **Batch Operations and Data Pipeline Processing** - Efficient batch concatenation, transformations, filtering, splitting, and aggregation operations
3. **Device Management and Memory Optimization** - Cross-device tensor operations, memory optimization techniques, data type management, and efficient transfers
4. **Neural Network Integration with TensorDictModule** - Seamless PyTorch integration with TensorDictModule and TensorDictSequential for structured data flow
5. **Advanced Data Structures and Reinforcement Learning Integration** - Complex RL data structures, experience replay buffers, temporal operations, and multi-agent environments

The tensordict package provides powerful data structures for organizing tensors with automatic batch dimension handling, enabling efficient ML workflows with complex, structured data.